In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field

loader=PyPDFLoader("../data/PdfData.pdf")
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splittedData=splitter.split_documents(docs)

embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

vector_db=InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embeddings
)

llm=ChatGroq(model="openai/gpt-oss-20b")


In [3]:
class RagState(BaseModel):
    question:str = Field(description="user question")
    documents:list = []
    context:str=Field(description="context data for user question",default="")
    answer:str = Field(description="Final answer...",default="")

In [4]:
def retrive_node(state:RagState):
    docs=vector_db.similarity_search(query=state.question)
    state.documents=docs
    return state

In [5]:
def create_context(state:RagState):
    context=""
    for doc in docs:
        context=context+doc.page_content+"\n\n"

    state.context=context
    return state

def generate_node(state:RagState):
    prompt=f"""
        you are a assistent and provide the answer for user question based on the provided context. If u don't find the relevent ans then just say i do not konw.
        context is:{state.context}
        question isL{state.question}
     """

    res=llm.invoke(prompt)
    state.answer=res.content
    return state

In [11]:
graph = StateGraph(RagState)
graph.add_node("retrive_node",retrive_node)
graph.add_node("create_context_node", create_context)
graph.add_node("generate_node",generate_node)

graph.add_edge(START,"retrive_node")
graph.add_edge("retrive_node","create_context_node")
graph.add_edge("create_context_node","generate_node")
graph.add_edge("generate_node",END)

graph=graph.compile()

res=graph.invoke({"question":"pm of india is"})

In [12]:
print(res["answer"])

I do not know.
